# Нормализующие потоки

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Нормализующие потоки».

## Что делает метод

Нормализующий поток — это обратимое преобразование, которое связывает простое базовое распределение (обычно стандартное нормальное) и сложное распределение данных. После обучения поток умеет делать два дела одновременно:

1. **Точно вычислять плотность вероятности** любой точки данных (в отличие от GAN, где это невозможно).
2. **Генерировать новые образцы**: достаточно взять точку из нормального распределения и применить обученное обратимое преобразование.

Для учёного это инструмент, который позволяет строить статистические модели сложных данных — распределений параметров симулятора, плотностей в химическом пространстве, профилей сигналов — и получать не просто точечную оценку, а полную плотность вероятности.

В этом блокноте мы построим небольшой поток в стиле **RealNVP** на PyTorch и обучим его на двумерном распределении в форме «двух лун». Блокнот исполняется на CPU за полминуты. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что данные — это облако точек в форме «двух лун»: нестандартное, не гауссово распределение. Нормализующий поток ищет обратимое преобразование $f$, которое «распрямляет» это облако в стандартное нормальное.

Если такое преобразование найдено, то:
- Плотность в исходном пространстве считается по **теореме о замене переменных**: $p(x) = p_z(f(x)) \cdot |\det J_f(x)|$, где $J_f$ — матрица Якоби преобразования.
- Генерация: берём $z \sim \mathcal{N}(0, I)$, применяем обратное преобразование $f^{-1}(z)$ — получаем новый реалистичный образец.

**Слой связывания (coupling layer)** — стандартный блок потоков — делает преобразование обратимым и с аналитически вычислимым якобианом:
1. Разбиваем вектор на две части $(x_1, x_2)$.
2. Первую часть оставляем неизменной: $y_1 = x_1$.
3. Вторую сдвигаем и масштабируем, используя первую как «ключ»: $y_2 = x_2 \cdot \exp(s(x_1)) + t(x_1)$, где $s$ и $t$ — произвольные нейросети.

Обратное преобразование тривиально: $x_2 = (y_2 - t(y_1)) \cdot \exp(-s(y_1))$. Якобиан — диагональный: $\log |\det J| = \sum s(x_1)$.

Стек таких слоёв с чередованием осей разбиения образует выразительный поток.

**Связь с выводом апостериора в симуляционных задачах:** нормализующие потоки лежат в основе современных методов *simulation-based inference* (SBI, нейросетевой байесовский вывод). Когда у нас есть сложный симулятор, задающий вероятность данных при известных параметрах, поток обучается аппроксимировать апостериорное распределение параметров по данным наблюдений — заменяя MCMC, который требует многократного запуска симулятора.

**Терминологический минимум:**
- **Плотность вероятности** — функция $p(x)$, показывающая, насколько вероятно наблюдение $x$.
- **Якобиан** — матрица частных производных преобразования; его определитель характеризует «растяжение» объёма при отображении.
- **Логарифмическое правдоподобие** — логарифм плотности; используется как функция потерь (максимизируем).
- **Базовое распределение** — простое распределение (обычно $\mathcal{N}(0, I)$), из которого начинается поток.

## 1. Установка библиотек

В Google Colab PyTorch, NumPy и scikit-learn уже установлены. Ячейка ниже гарантирует нужные версии и при локальном запуске.

In [ ]:
%pip install -q torch==2.3.1 numpy matplotlib scikit-learn

## 2. Единый стиль графиков

Все графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Синтетические данные: «две луны»

Используем двумерное распределение «двух лун» из `scikit-learn`. Это стандартный тест для методов оценки плотности: распределение несепарабельное, нелинейное, непохожее на нормальное. Никаких загрузок не требуется — данные генерируются в памяти.

Зашумлённость (`noise=0.05`) добавляет небольшое размытие, делая задачу более реалистичной.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# Фиксируем генераторы случайных чисел для воспроизводимости.
np.random.seed(42)
torch.manual_seed(42)

N_SAMPLES = 800
data_np, _ = make_moons(n_samples=N_SAMPLES, noise=0.05, random_state=42)
data_np = data_np.astype(np.float32)

# Центрируем и нормируем для удобства обучения.
data_np = (data_np - data_np.mean(0)) / data_np.std(0)

data_tensor = torch.from_numpy(data_np)  # (N_SAMPLES, 2)

# Предварительная визуализация данных.
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(data_np[:, 0], data_np[:, 1],
           color=VIZ["series"][0], s=10, alpha=0.6)
ax.set_title("Исходные данные: распределение 'двух лун'")
ax.set_xlabel("Признак 1")
ax.set_ylabel("Признак 2")
fig.tight_layout()
plt.show()

print(f"Обучающих точек: {N_SAMPLES}, размерность: 2")

## 4. Применение метода

### Шаг 1. Слой связывания (CouplingLayer)

Каждый слой связывания делит двумерный вектор на две части по паритету измерения (`mask`): чётные координаты обрабатываются одним способом, нечётные — другим. В двумерном случае каждый слой фиксирует одну координату и преобразует другую.

Нейросети `s_net` и `t_net` внутри слоя — это простые двухслойные MLP. Они принимают зафиксированную часть и вычисляют параметры масштаба и сдвига для трансформируемой части.

Метод `forward` — прямое преобразование (данные → латентное пространство, для обучения).  
Метод `inverse` — обратное преобразование (латентное пространство → данные, для генерации).

In [ ]:
import torch.nn as nn


class CouplingLayer(nn.Module):
    """Слой связывания RealNVP для двумерных данных."""

    def __init__(self, mask, hidden=32):
        """mask: булев тензор (dim,) — True = зафиксированные координаты."""
        super().__init__()
        self.register_buffer("mask", mask.float())
        dim = mask.shape[0]
        # s_net выдаёт log-масштаб, t_net — сдвиг.
        self.s_net = nn.Sequential(
            nn.Linear(dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, dim),
        )
        self.t_net = nn.Sequential(
            nn.Linear(dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x):
        """x -> z. Возвращает (z, log|det J|)."""
        x_masked = x * self.mask                        # зафиксированная часть
        s = self.s_net(x_masked) * (1 - self.mask)     # только по трансформируемым
        t = self.t_net(x_masked) * (1 - self.mask)
        z = x_masked + (1 - self.mask) * (x * torch.exp(s) + t)
        log_det = s.sum(dim=1)                          # сумма по трансформируемым
        return z, log_det

    def inverse(self, z):
        """z -> x. Обратное преобразование (аналитическое)."""
        z_masked = z * self.mask
        s = self.s_net(z_masked) * (1 - self.mask)
        t = self.t_net(z_masked) * (1 - self.mask)
        x = z_masked + (1 - self.mask) * ((z - t) * torch.exp(-s))
        return x

### Шаг 2. Полная модель потока

Стекируем несколько слоёв связывания с чередующимися масками: в чётных слоях фиксируем первую координату, в нечётных — вторую. Такое чередование гарантирует, что обе координаты «видят» друг друга через достаточное число слоёв.

**Обучение** (метод `log_prob`): применяем цепочку прямых преобразований, накапливаем `log|det J|`, считаем логарифм плотности базового распределения и получаем точное логарифмическое правдоподобие.

**Генерация** (метод `sample`): семплируем из базового нормального распределения и применяем обратные преобразования в обратном порядке.

In [ ]:
class RealNVP(nn.Module):
    """Нормализующий поток RealNVP: стек слоёв связывания."""

    def __init__(self, dim=2, n_layers=6, hidden=32):
        super().__init__()
        # Чередуем маски: [True, False] и [False, True]
        masks = []
        for i in range(n_layers):
            m = torch.zeros(dim, dtype=torch.bool)
            m[i % dim] = True
            masks.append(m)
        self.layers = nn.ModuleList([
            CouplingLayer(m, hidden=hidden) for m in masks])
        # Базовое распределение: стандартное нормальное N(0, I)
        self.register_buffer("base_mean", torch.zeros(dim))
        self.register_buffer("base_std",  torch.ones(dim))

    def log_prob(self, x):
        """Точное логарифмическое правдоподобие для каждой точки."""
        log_det_total = torch.zeros(x.shape[0], device=x.device)
        z = x
        for layer in self.layers:
            z, log_det = layer(z)
            log_det_total = log_det_total + log_det
        # log p(z) по нормальному распределению
        log_pz = (-0.5 * z.pow(2) - 0.5 * torch.log(torch.tensor(2 * torch.pi))).sum(dim=1)
        return log_pz + log_det_total

    @torch.no_grad()
    def sample(self, n):
        """Генерирует n точек из обученного распределения."""
        z = torch.randn(n, self.base_mean.shape[0], device=self.base_mean.device)
        x = z
        for layer in reversed(self.layers):
            x = layer.inverse(x)
        return x


model = RealNVP(dim=2, n_layers=6, hidden=32)
n_params = sum(p.numel() for p in model.parameters())
print(f"Слоёв связывания: 6")
print(f"Параметров в модели: {n_params}")

### Шаг 3. Обучение

Функция потерь — отрицательное среднее логарифмическое правдоподобие (NLL). Её минимизация эквивалентна максимизации правдоподобия модели при наблюдении данных.

Обратите внимание: это **не** аппроксимация и не вариационная нижняя граница — поток вычисляет **точное** правдоподобие, используя теорему о замене переменных с аналитически считаемым якобианом.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 128
N_EPOCHS   = 40

ds     = TensorDataset(data_tensor)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
opt    = torch.optim.Adam(model.parameters(), lr=5e-3)
sched  = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.5)

history = []
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_nll = 0.0
    for (xb,) in loader:
        opt.zero_grad()
        nll = -model.log_prob(xb).mean()   # минимизируем NLL
        nll.backward()
        # Ограничиваем норму градиентов для стабильности обучения.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        ep_nll += nll.item() * len(xb)
    sched.step()
    history.append(ep_nll / N_SAMPLES)
    if epoch % 8 == 0:
        print(f"Эпоха {epoch:3d}: средний NLL = {history[-1]:.4f}")

print("Обучение завершено.")

### Шаг 4. Визуализация результатов

Строим два графика на одной фигуре:

1. **Образцы потока поверх обучающих данных**: генерируем точки из обученного распределения и накладываем на исходные данные. Хорошая модель: синтетические точки должны заполнять ту же область «двух лун».
2. **Тепловая карта плотности**: вычисляем $\log p(x)$ на равномерной сетке и строим тепловую карту. Это прямое преимущество потоков перед GAN: точная плотность доступна везде.

In [ ]:
model.eval()

# Генерируем образцы из обученного потока.
samples = model.sample(800).numpy()

# Вычисляем плотность на сетке для тепловой карты.
res = 80
x1_range = np.linspace(-2.5, 2.5, res, dtype=np.float32)
x2_range = np.linspace(-2.0, 2.0, res, dtype=np.float32)
xx1, xx2 = np.meshgrid(x1_range, x2_range)
grid_pts  = np.stack([xx1.ravel(), xx2.ravel()], axis=1)  # (res*res, 2)
grid_t    = torch.from_numpy(grid_pts)

with torch.no_grad():
    log_p = model.log_prob(grid_t).numpy()   # (res*res,)

log_p_grid = log_p.reshape(res, res)
# Для отображения заменяем -inf на минимальное конечное значение.
finite_min = log_p_grid[np.isfinite(log_p_grid)].min() if np.isfinite(log_p_grid).any() else -30.0
log_p_grid = np.where(np.isfinite(log_p_grid), log_p_grid, finite_min)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

# --- Левый: данные и образцы ---
axes[0].scatter(data_np[:, 0], data_np[:, 1],
                color=VIZ["series"][3], s=10, alpha=0.5,
                label="обучающие данные", zorder=2)
axes[0].scatter(samples[:, 0], samples[:, 1],
                color=VIZ["series"][0], s=10, alpha=0.5,
                label="образцы из потока", zorder=3)
axes[0].set_title("Данные и образцы обученного потока")
axes[0].set_xlabel("Признак 1")
axes[0].set_ylabel("Признак 2")
axes[0].legend(loc="upper right")

# --- Правый: тепловая карта log-плотности ---
im = axes[1].contourf(xx1, xx2, log_p_grid, levels=30,
                      cmap="Blues")
axes[1].scatter(data_np[:, 0], data_np[:, 1],
                color=VIZ["series"][3], s=8, alpha=0.35,
                label="данные", zorder=3)
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04,
             label="log p(x)")
axes[1].set_title("Тепловая карта логарифмической плотности")
axes[1].set_xlabel("Признак 1")
axes[1].set_ylabel("Признак 2")
axes[1].legend(loc="upper right")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый — наложение образцов на данные: серые точки — исходные данные, синие — сгенерированные потоком. Если они перекрываются и занимают одну область пространства — поток выучил правильное распределение. Если синие точки разбросаны хаотично или сосредоточены только в одной «луне» — нужно больше слоёв или эпох.

Правый — тепловая карта: чем темнее синий, тем выше логарифмическая плотность $\log p(x)$. Точки данных наложены поверх карты: они должны попадать в тёмные области. Светлые «пустые» области между лунами должны иметь низкую плотность. Это демонстрирует главное преимущество потоков: мы можем вычислить $p(x)$ в любой точке пространства — не только там, где есть данные.

### Шаг 5. Кривая обучения и базовое распределение

Покажем, как выглядят точки данных **в латентном пространстве** — то есть после применения всех прямых преобразований. Если поток обучился хорошо, они должны образовывать облако, близкое к стандартному нормальному. Это наглядно иллюстрирует «нормализующую» часть названия метода.

In [ ]:
# Прямое преобразование данных в латентное пространство.
model.eval()
with torch.no_grad():
    z_all = data_tensor.clone()
    for layer in model.layers:
        z_all, _ = layer(z_all)
z_all = z_all.numpy()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

# --- Левый: данные в латентном пространстве ---
axes[0].scatter(z_all[:, 0], z_all[:, 1],
                color=VIZ["series"][1], s=12, alpha=0.6)
# Контур стандартного нормального для сравнения.
theta = np.linspace(0, 2 * np.pi, 200)
for r in (1, 2):
    axes[0].plot(r * np.cos(theta), r * np.sin(theta),
                 color=VIZ["edge"], linewidth=1.0, linestyle="--", alpha=0.6)
axes[0].set_title("Данные в латентном пространстве (после потока)")
axes[0].set_xlabel("Латентная ось 1")
axes[0].set_ylabel("Латентная ось 2")
axes[0].set_aspect("equal")

# --- Правый: кривая обучения ---
axes[1].plot(range(1, N_EPOCHS + 1), history, color=VIZ["series"][0])
axes[1].set_title("Сходимость обучения потока")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Среднее отрицательное логарифмическое правдоподобие")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый — данные в латентном пространстве: пунктирные окружности показывают радиусы 1 и 2 стандартного нормального распределения (охватывают ~68% и ~95% нормального облака). Если зелёные точки образуют примерно круглое облако, сосредоточенное около начала координат и укладывающееся в эти окружности, — поток успешно «нормализовал» исходное нестандартное распределение. Остатки структуры «лун» говорят о недостаточной выразительности модели.

Правый — кривая обучения: убывание NLL означает улучшение модели. После расписания шага (~15-я и ~30-я эпохи) кривая может делать небольшие скачки вниз.

## 5. Интерпретация результата

- **Тепловая карта плотности** показывает, где модель оценивает высокую вероятность. Хороший поток: пики плотности совпадают с «лунами»; зазор между ними — низкая плотность.
- **Образцы против данных**: синтетические точки должны занимать ту же область, что и исходные. Небольшое «расплывание» допустимо при малом числе слоёв.
- **Латентное пространство**: после всех преобразований данные должны выглядеть как стандартный нормальный шар. Чем ближе к нему — тем лучше поток выполнил нормализацию.
- **NLL как мера качества**: для сравнения двух потоков на одних данных берут модель с меньшим NLL. Значение само по себе труднее интерпретировать абсолютно — важна динамика убывания.
- **Выбор числа слоёв**: 4–8 слоёв обычно достаточно для 2D задач. Для реальных данных высокой размерности нужно больше слоёв или более выразительные $s$- и $t$-сети.
- **Связь с Байесовским выводом**: в simulation-based inference поток обучается аппроксимировать $p(\theta | \text{данные})$ — апостериорное распределение параметров симулятора. Это открывает возможность вывода апостериора для «чёрноящичных» симуляторов без аналитической формулы правдоподобия.

## 6. Подставьте свои данные

Нормализующий поток применим к любым непрерывным многомерным данным. Типичные задачи в науке:

- Апостериорное распределение параметров модели, полученное с помощью MCMC.
- Плотность в химическом или конформационном пространстве молекул.
- Совместное распределение нескольких измеримых величин эксперимента.

Для данных размерности выше 4–8 рекомендуется увеличить `hidden` (скрытые слои нейросетей $s$ и $t$) и `n_layers` (число слоёв связывания).

In [ ]:
# --- Шаблон загрузки своих данных ---
# Предполагается, что у вас есть матрица X формы (N, D),
# где D — число непрерывных признаков.
#
# import pandas as pd
# import numpy as np
# import torch
#
# X = pd.read_csv('my_data.csv').to_numpy(dtype='float32')
# # Центрируем и нормируем каждый признак:
# X = (X - X.mean(0)) / (X.std(0) + 1e-8)
# data_tensor = torch.from_numpy(X)
#
# DIM = X.shape[1]
# print(f'Точек: {len(X)}, размерность: {DIM}')
#
# # Создайте модель нужной размерности:
# # my_model = RealNVP(dim=DIM, n_layers=8, hidden=64)
#
# # Затем повторите ячейки раздела 4 (обучение и визуализация).
# # Для D > 2 тепловую карту стройте для пары признаков, маргинализируя остальные.

## 7. Поэкспериментируйте сами

Измените один параметр, перезапустите разделы 3–4 и сравните результат.

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `n_layers` | Уменьшить до 2 | Латентное облако хуже нормализуется; образцы хуже воспроизводят «луны» |
| `n_layers` | Увеличить до 10 | Более точная плотность, медленнее обучение |
| `hidden` | Уменьшить до 16 | Быстрее, но менее выразительные преобразования |
| `noise` | Увеличить до 0.2 в `make_moons` | Данные размытее; поток должен давать более широкую тепловую карту |
| Данные | Заменить на двумерную смесь гауссианов: `np.vstack([np.random.randn(400, 2) + [2,2], np.random.randn(400, 2) + [-2,-2]])` | Более простая задача — проверьте, быстрее ли сходится |

Важное наблюдение: при очень малом `n_layers` латентное облако сохраняет форму «лун» — поток не справился с нормализацией. При достаточном числе слоёв облако становится круглым — нормализация достигнута.

## Краткие выводы

- Нормализующий поток — обратимое нейросетевое преобразование, которое связывает простое базовое распределение со сложным распределением данных.
- Ключевое преимущество перед GAN и VAE: **точное логарифмическое правдоподобие** доступно для любой точки данных через теорему о замене переменных.
- Слой связывания (coupling layer) обеспечивает аналитически обратимое преобразование с диагональным якобианом — это и делает вычисление $\log p(x)$ эффективным.
- **Генерация**: семплировать из нормального распределения и применить обратное преобразование — быстро и без итеративных процедур.
- **Связь с наукой**: потоки лежат в основе нейросетевого байесовского вывода (SBI/NPE): они аппроксимируют апостериорное распределение параметров симулятора по данным наблюдений, заменяя многократные запуски MCMC.
- Ограничения: требуют непрерывных данных; для дискретных величин нужны дискретные потоки или дополнительные техники.